# EasyVisa: Predicting Visa Application Status

## 1\. Problem Definition and Objective

**Context:** Business communities in the United States are facing high demand for human resources but struggle with the challenge of identifying and attracting the right talent, both locally and from abroad. The Immigration and Nationality Act (INA) allows US employers to hire foreign workers on a temporary or permanent basis, a process overseen by the Office of Foreign Labor Certification (OFLC). The OFLC processes job certification applications, granting them when employers can demonstrate that no sufficient US workers are available for the role at a fair wage.

**Problem:** The number of visa applications is increasing significantly each year, making the manual review of every case a tedious and time-consuming task for the OFLC. The firm "EasyVisa" has been hired to provide a data-driven solution to this problem.

**Objective:** The primary goal of this project is to build a classification model that can help EasyVisa in the following ways:

  1. **Facilitate the process of visa approvals:** By accurately predicting the outcome of a visa application (Certified or Denied).
  2. **Recommend a suitable profile for applicants:** By identifying the key drivers and factors that significantly influence the visa certification status. This will help employers understand what constitutes a strong application.

## 2\. Data Dictionary

The dataset contains various attributes of the employee and the employer.

Feature | Description | Data Type  
---|---|---  
`case_id` | Unique ID of each visa application | Character  
`continent` | Continent of the employee's origin | Character  
`education_of_employee` | Education level of the employee | Character  
`has_job_experience` | Does the employee have any job experience? (Y/N) | Character  
`requires_job_training` | Does the employee require any job training? (Y/N) | Character  
`no_of_employees` | Number of employees in the employer's company | Integer  
`yr_of_estab` | Year in which the company was established | Integer  
`region_of_employment` | The region of the employer's intended work location | Character  
`prevailing_wage` | The prevailing wage for the job in the intended area | Float  
`unit_of_wage` | The unit of the prevailing wage (Hour, Week, Month, Year) | Character  
`full_time_position` | Is the position full-time? (Y/N) | Character  
`case_status` | **Target Variable:** Visa application status (Certified/Denied) | Character

## 3\. Importing Necessary Libraries

In [ ]:
# Let's start by importing all the essential Python libraries for data manipulation, visualization, modeling, and evaluation.

#Basic data manipulation and numerical operations
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# To set plotting styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)

# Data preprocessing and feature engineering
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Imbalanced data handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Machine Learning Models
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
import xgboost as xgb
import imblearn
from xgboost import XGBClassifier

# Model evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

# To ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Imbalanced-learn version:", imblearn.__version__)
print("XGBoost version:", xgb.__version__)

## 4\. Data Loading and Initial Exploration

In [ ]:
# Understanding the structure of the data
# Read the data
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/EasyVisa.csv')

print("\nShape of the dataset (rows, columns):", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())
print("\nDataset Information:")
df.info()

## 5\. Data Pre-processing (As per Rubric)

**5.1. Missing Value Detection and Treatment**

In [ ]:
# Let's check for any missing values in the dataset.

print("\nMissing values in each column:")
print(df.isnull().sum())

**Rationale & Treatment:** The output shows that there are **no missing values** in any of the columns. This is excellent, as it means we do not need to perform imputation or drop any rows/columns due to missing data. The dataset is clean in this regard.

**5.2. Dropping Irrelevant Columns**

In [ ]:
# The `case_id` column is a unique identifier for each application. It holds no predictive power and is irrelevant for a classification model. We will drop it.

df_processed = df.drop('case_id', axis=1)
print("\n'case_id' column dropped.")

**5.3. Outlier Detection and Treatment**

In [ ]:
# We will check for outliers in the numerical columns: `no_of_employees`, `yr_of_estab`, and `prevailing_wage`. We'll use boxplots for visualization.

numerical_cols = ['no_of_employees', 'yr_of_estab', 'prevailing_wage']
plt.figure(figsize=(18, 5))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df_processed[col])
    plt.title(f'Boxplot of {col}')
plt.suptitle('Outlier Detection in Numerical Columns', fontsize=16)
plt.show()

**Rationale & Treatment:**

  * **`no_of_employees` & `prevailing_wage`:** Both show a significant number of high-value outliers. These are not necessarily errors. They could represent very large companies or very high-paying specialized jobs, which could be important predictors of visa success. Removing them would mean losing valuable information.
  * **`yr_of_estab`:** Shows some low-value outliers, representing very old companies. Again, this is legitimate data.



**Decision:** Decision Trees and ensemble models based on them (like Random Forest) are robust to outliers because they split data based on thresholds, not on the magnitude of the values. Therefore, **we will not treat these outliers** , as they represent potentially important segments of the data.

**5.4. Feature Engineering**

In [ ]:
# We can create a new feature, `company_age`, from `yr_of_estab`. A company's age might be a better predictor of its stability and ability to sponsor a visa than its establishment year alone. We'll assume the data is from the current year (2016, as mentioned in the problem description).

df_processed['company_age'] = 2016 - df_processed['yr_of_estab']
# We can now drop the original 'yr_of_estab' column
df_processed = df_processed.drop('yr_of_estab', axis=1)
print("\nFeature 'company_age' created and 'yr_of_estab' dropped.")
print(df_processed[['company_age']].head())

**5.5. Preparing Data for Modeling**

In [ ]:
# Machine learning models require numerical input. We need to convert our categorical features into a numerical format.
#
# - **Binary Features:** `has_job_experience`, `requires_job_training`, `full_time_position`. We can map 'Y' to 1 and 'N' to 0.
# - **Target Variable:** `case_status`. We'll map 'Certified' to 1 and 'Denied' to 0.
# - **Multi-level Categorical Features:** `continent`, `education_of_employee`, `region_of_employment`, `unit_of_wage`. We will use **one-hot encoding** (using `pd.get_dummies`) for these to avoid creating an artificial ordinal relationship.

# Mapping binary features
binary_map = {'Y': 1, 'N': 0}
df_processed['has_job_experience'] = df_processed['has_job_experience'].map(binary_map)
df_processed['requires_job_training'] = df_processed['requires_job_training'].map(binary_map)
df_processed['full_time_position'] = df_processed['full_time_position'].map(binary_map)

# Mapping target variable
df_processed['case_status'] = df_processed['case_status'].map({'Certified': 1, 'Denied': 0})

# One-hot encoding for multi-level categorical features
df_model_ready = pd.get_dummies(df_processed, drop_first=True)

print("\nShape of the final model-ready data:", df_model_ready.shape)
print("\nFirst 5 rows of the final model-ready data:")
print(df_model_ready.head())

# 6\. Exploratory Data Analysis (EDA) (as per Rubric)

We will now perform a detailed exploration of the data to identify patterns, relationships, and key insights.

**6.1. Detailed Univariate Analysis**

In [ ]:
# Here, we explore each variable individually to understand its distribution in detail.
#
# #### **Target Variable: `case_status`**
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='case_status', data=df_processed, palette='viridis')
plt.title('Distribution of Visa Case Status', fontsize=16)
plt.xlabel('Case Status')
plt.ylabel('Number of Applications')
plt.xticks([0, 1], ['Denied', 'Certified'])
total = len(df_processed)
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width() / 2., height + 100, f'{height/total*100:.2f}%', ha="center", fontsize=12)
plt.show()

**Observation:** The dataset is imbalanced. Approximately **66.78% of applications were certified** , while **33.22% were denied**. This 2:1 ratio is important to consider, as models might become biased towards the majority class ('Certified').

In [ ]:
#Categorical Variables
cat_cols = ['continent', 'education_of_employee', 'has_job_experience', 'requires_job_training', 'region_of_employment', 'unit_of_wage', 'full_time_position']
for col in cat_cols:
    plt.figure(figsize=(12, 7))
    ax = sns.countplot(y=col, data=df_processed, order=df_processed[col].value_counts().index, palette='mako')
    plt.title(f'Distribution of Applicants by {col}', fontsize=16)
    plt.xlabel('Number of Applications')
    plt.ylabel(col.replace('_', ' ').title())
    total = len(df_processed)
    for p in ax.patches:
        width = p.get_width()
        plt.text(50 + width, p.get_y() + 0.55 * p.get_height(),
                 f'{width/total*100:.2f}%',
                 ha='center', va='center')
    plt.show()

**Observations on Categorical Variables:**

  * **Continent:** The vast majority of applicants (**66.1 continente**) are from **Asia** , followed by Europe (14.65%) and North America (12.92%).
  * **Education:** Most applicants hold a **Bachelor's (40.16%)** , **Master's (37.81%)** , or **High School (13.49%)** degree. Doctorates are less common (8.54%).
  * **Job Experience:** A significant majority (**58.08%**) of applicants have prior job experience.
  * **Job Training:** Most applicants (**88.4requires_job_trainingo not require job training for the position.
  * **Region of Employment:** The most popular regions for employment are the **Northeast (28.24%)** , **South (27.56%)** , and **West (25.85%)**.
  * **Unit of Wage:** The overwhelming majority (**90.1 unit_of_wagee paid annually ('Year'), suggesting most applications are for salaried, professional roles.
  * **Full-Time Position:** Almost all applications (**89.37%**) are for full-time positions.

In [ ]:
#Numerical Variables
# We will use both histograms (for distribution shape) and boxplots (for spread and outliers).
num_cols = ['no_of_employees', 'prevailing_wage', 'company_age']
for col in num_cols:
    plt.figure(figsize=(16, 6))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(df_processed[col], bins=40, kde=True)
    plt.title(f'Distribution of {col}', fontsize=14)

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df_processed[col])
    plt.title(f'Boxplot of {col}', fontsize=14)

    plt.suptitle(f'Univariate Analysis of {col.replace("_", " ").title()}', fontsize=18)
    plt.show()

**Observations on Numerical Variables:**

  * **Number of Employees:** The distribution is heavily right-skewed. Most companies are small to medium-sized, with a median number of employees around 2,100. There is a very long tail of large corporations, which appear as outliers.
  * **Prevailing Wage:** This is also strongly right-skewed. The median wage is approximately \$70,000, but there are many high-paying positions that extend far into the upper range, indicating specialized, high-skill jobs.
  * **Company Age:** The distribution of company age is left-skewed, which is interesting. It shows a large number of established, older companies, with a median age of around 37 years. There are fewer very young companies in the dataset.

**6.2. Detailed Bivariate and Multivariate Analysis**

Now we'll explore the relationships between variables, focusing on how they interact with the `case_status`.

In [ ]:
#Categorical Features vs. Case Status
for col in cat_cols:
    plt.figure(figsize=(12, 8))
    sns.countplot(y=col, data=df_processed, hue='case_status', order=df_processed[col].value_counts().index, palette='coolwarm')
    plt.title(f'{col.replace("_", " ").title()} vs. Visa Status', fontsize=16)
    plt.xlabel('Number of Applications')
    plt.ylabel(col.replace('_', ' ').title())
    plt.legend(title='Case Status', labels=['Denied', 'Certified'])
    plt.show()

**Key Bivariate Observations (Categorical):**

  * **Education:** The certification rate clearly increases with education level. **Doctorates** have the highest approval rate, while High School graduates have the lowest.
  * **Job Experience:** Applicants **with job experience** have a dramatically higher chance of getting their visa certified compared to those without.
  * **Unit of Wage:** Applications with a **Yearly** wage unit have a far higher certification rate than those with Hourly, Weekly, or Monthly wages. This is a very strong indicator of a successful application.

In [ ]:
#Numerical Features vs. Case Status
for col in num_cols:
    plt.figure(figsize=(12, 7))
    sns.violinplot(x='case_status', y=col, data=df_processed, palette='viridis', inner='quartile')
    plt.title(f'{col.replace("_", " ").title()} vs. Visa Status', fontsize=16)
    plt.xlabel('Case Status')
    plt.ylabel(col.replace("_", " ").title())
    plt.xticks([0, 1], ['Denied', 'Certified'])
    plt.show()

**Key Bivariate Observations (Numerical):**

  * **Prevailing Wage:** While the overall distributions overlap significantly, the median wage for **certified** applications is noticeably higher than for denied ones. This suggests that higher-paying jobs have a better chance of approval.
  * **Number of Employees & Company Age:** The distributions for certified and denied applications are nearly identical for these two features. This implies that the size or age of the sponsoring company has very little direct impact on the visa outcome on its own.

**Multivariate Analysis**

Let's explore the interaction between multiple important variables.

In [ ]:
# Prevailing Wage, Education, and Visa Status
# How does the wage distribution look across different education levels for certified vs. denied cases?

plt.figure(figsize=(16, 9))
sns.boxenplot(x='education_of_employee', y='prevailing_wage', hue='case_status', data=df_processed,
              palette='coolwarm', order=["High School", "Bachelor's", "Master's", "Doctorate"])
plt.title('Prevailing Wage vs. Education by Visa Status', fontsize=18)
plt.xlabel('Education Level of Employee')
plt.ylabel('Prevailing Wage (Log Scale)')
plt.yscale('log') # Use a log scale to better visualize the skewed wage data
plt.legend(title='Case Status', labels=['Denied', 'Certified'])
plt.show()

**Multivariate Inference:**

  1. **Clear Wage Gap:** Within every education level, the median prevailing wage for **certified** applications is consistently higher than for denied applications.
  2. **Education's Amplifying Effect:** The wage gap between certified and denied applications becomes more pronounced as the education level increases.
  3. **The "Sweet Spot":** The combination of a **high education level (Master's or Doctorate) and a high prevailing wage** appears to be a strong recipe for a successful visa application. This is likely the most important interaction in the dataset.

**6.3. Correlation Analysis**

In [ ]:
# Let's visualize the correlation between all the final numerical features in our model-ready dataset.
plt.figure(figsize=(20, 15))
sns.heatmap(df_model_ready.corr(), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Matrix of All Features', fontsize=18)
plt.show()

**Correlation Inference:**

  * **`case_status`:** The target variable shows the strongest positive correlations with `has_job_experience`, `education_of_employee_Master's`, and `education_of_employee_Doctorate`. It has a strong negative correlation with `education_of_employee_High School` and `unit_of_wage_Hour`. This numerically confirms all our visual findings.
  * There is no strong multicollinearity between the independent features, which is good for modeling.

# Summary of EDA - Key Insights for Modeling:

  * The dataset is moderately imbalanced.
    * **Strongest Predictors of Success:** High education level, prior job experience, and a yearly (salaried) wage contract.
    * **Interaction is Key:** The combination of high education and high prevailing wage is a powerful predictor.
    * **Weak Predictors:** Company size and age appear to have minimal direct impact.

# 7\. Data Splitting

In [ ]:
# We will split the data into training (70%) and testing (30%) sets. We use `stratify` to maintain the same proportion of target classes in both sets, which is important for our imbalanced data.

X = df_model_ready.drop('case_status', axis=1)
y = df_model_ready['case_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# 8\. Model Building - Original Data (as per Rubric)

In [ ]:
# We will build 5 classification models on the original, imbalanced training data. We'll create a helper function to evaluate and store the results.

# Dictionary to store model results
results = {}

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Fits, predicts, and evaluates a model, then stores the results.
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results[model_name] = {'Accuracy': accuracy, 'Recall': recall, 'Precision': precision, 'F1-Score': f1}

    print(f"--- {model_name} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("-"*(len(model_name)+8) + "\n")

# Initialize models
dt_orig = DecisionTreeClassifier(random_state=42)
rf_orig = RandomForestClassifier(random_state=42)
bag_orig = BaggingClassifier(random_state=42)
ada_orig = AdaBoostClassifier(random_state=42)
gb_orig = GradientBoostingClassifier(random_state=42)
xgb_orig = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

# Evaluate models on original data
print("="*40)
print("  MODELING ON ORIGINAL (IMBALANCED) DATA")
print("="*40 + "\n")
evaluate_model(dt_orig, X_train, y_train, X_test, y_test, 'Decision Tree (Original)')
evaluate_model(rf_orig, X_train, y_train, X_test, y_test, 'Random Forest (Original)')
evaluate_model(bag_orig, X_train, y_train, X_test, y_test, 'Bagging (Original)')
evaluate_model(ada_orig, X_train, y_train, X_test, y_test, 'AdaBoost (Original)')
evaluate_model(gb_orig, X_train, y_train, X_test, y_test, 'Gradient Boosting (Original)')
evaluate_model(xgb_orig, X_train, y_train, X_test, y_test, 'XGBoost (Original)')

# 9\. Handling Class Imbalance

In [ ]:
# The original data has a 67/33 split. This can bias models. We will now try two techniques to balance the training data:
# 1.  Oversampling (SMOTE):** Synthetically creates new minority class instances.
# 2.  Undersampling (RandomUnderSampler):** Randomly removes majority class instances.
#
# Important: We only apply these techniques to the **training data** to avoid data leakage into the test set.
#
# 9.1. Oversampling with SMOTE

print("\nOriginal training data distribution:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nSMOTE-oversampled training data distribution:")
print(y_train_smote.value_counts())

# ### 9.2. Undersampling with RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("\nRandomly-undersampled training data distribution:")
print(y_train_rus.value_counts())

# 10\. Model Building - Oversampled Data (as per Rubric)

In [ ]:
# Now, we build the same 5+ models on the SMOTE-oversampled training data.

dt_smote = DecisionTreeClassifier(random_state=42)
rf_smote = RandomForestClassifier(random_state=42)
bag_smote = BaggingClassifier(random_state=42)
ada_smote = AdaBoostClassifier(random_state=42)
gb_smote = GradientBoostingClassifier(random_state=42)
xgb_smote = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

print("\n" + "="*40)
print("  MODELING ON OVERSAMPLED (SMOTE) DATA")
print("="*40 + "\n")
evaluate_model(dt_smote, X_train_smote, y_train_smote, X_test, y_test, 'Decision Tree (SMOTE)')
evaluate_model(rf_smote, X_train_smote, y_train_smote, X_test, y_test, 'Random Forest (SMOTE)')
evaluate_model(bag_smote, X_train_smote, y_train_smote, X_test, y_test, 'Bagging (SMOTE)')
evaluate_model(ada_smote, X_train_smote, y_train_smote, X_test, y_test, 'AdaBoost (SMOTE)')
evaluate_model(gb_smote, X_train_smote, y_train_smote, X_test, y_test, 'Gradient Boosting (SMOTE)')
evaluate_model(xgb_smote, X_train_smote, y_train_smote, X_test, y_test, 'XGBoost (SMOTE)')

# 11\. Model Building - Undersampled Data (as per Rubric)

In [ ]:
# Next, we build the models on the randomly undersampled training data.

dt_rus = DecisionTreeClassifier(random_state=42)
rf_rus = RandomForestClassifier(random_state=42)
bag_rus = BaggingClassifier(random_state=42)
ada_rus = AdaBoostClassifier(random_state=42)
gb_rus = GradientBoostingClassifier(random_state=42)
xgb_rus = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

print("\n" + "="*40)
print("  MODELING ON UNDERSAMPLED (RUS) DATA")
print("="*40 + "\n")
evaluate_model(dt_rus, X_train_rus, y_train_rus, X_test, y_test, 'Decision Tree (RUS)')
evaluate_model(rf_rus, X_train_rus, y_train_rus, X_test, y_test, 'Random Forest (RUS)')
evaluate_model(bag_rus, X_train_rus, y_train_rus, X_test, y_test, 'Bagging (RUS)')
evaluate_model(ada_rus, X_train_rus, y_train_rus, X_test, y_test, 'AdaBoost (RUS)')
evaluate_model(gb_rus, X_train_rus, y_train_rus, X_test, y_test, 'Gradient Boosting (RUS)')
evaluate_model(xgb_rus, X_train_rus, y_train_rus, X_test, y_test, 'XGBoost (RUS)')

# 12\. Hyperparameter Tuning (as per Rubric)

**12.1. Choosing the Best Performing Models**

In [ ]:
# Let's compile the results to choose the top 3 models for tuning. We will focus on the **F1-Score** as our primary metric because it provides a balance between Precision and Recall, which is ideal for imbalanced datasets.

results_df = pd.DataFrame(results).T.sort_values(by='F1-Score', ascending=False)
print("\n--- Model Performance Comparison ---")
print(results_df)

**Reason for Choices:**

Based on the F1-scores from your notebook, the top-performing models are consistently the ensemble methods. We will choose the following three for hyperparameter tuning:

  1. **Gradient Boosting (Original):** It has the highest F1-score overall in the initial run.
  2. **XGBoost (Original):** This model also performed very well on the original data, showing a high F1-score.
  3. **Random Forest (Original):** Another good performer, providing a solid baseline for comparison.

**12.2. Tuning the Chosen Models with GridSearchCV**

In [ ]:
# We will now perform an exhaustive search for the best hyperparameters for our chosen models.

# Helper function for tuning
def tune_model(model, param_grid, X_train, y_train, model_name):
    print(f"\n--- Tuning {model_name} ---")
    grid_search = GridSearchCV(estimator=model,
                               param_grid=param_grid,
                               cv=3, # 3-fold cross-validation
                               scoring='f1',
                               verbose=1,
                               n_jobs=-1) # Use all available cores
    grid_search.fit(X_train, y_train)
    print(f"Best F1-Score from CV: {grid_search.best_score_:.4f}")
    print("Best Hyperparameters:")
    print(grid_search.best_params_)
    return grid_search.best_estimator_

# --- Tuning XGBoost ---
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8, 1.0]
}
tuned_xgb = tune_model(XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False), xgb_params, X_train, y_train, 'XGBoost')

# --- Tuning Random Forest ---
rf_params = {
    'n_estimators': [150, 250],
    'max_depth': [10, 15],
    'min_samples_leaf': [3, 5],
    'min_samples_split': [5, 10]
}
tuned_rf = tune_model(RandomForestClassifier(random_state=42), rf_params, X_train, y_train, 'Random Forest')

# --- Tuning Gradient Boosting ---
gb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8, 1.0]
}
tuned_gb = tune_model(GradientBoostingClassifier(random_state=42), gb_params, X_train, y_train, 'Gradient Boosting')

# 13\. Model Performances - Final Comparison (as per Rubric)

**13.1. Performance on Test Data**

In [ ]:
# Now we evaluate our newly tuned models on the unseen test data and compare them to select the final, champion model.

print("\n" + "="*40)
print("      EVALUATING TUNED MODELS")
print("="*40 + "\n")
evaluate_model(tuned_xgb, X_train, y_train, X_test, y_test, 'XGBoost (Tuned)')
evaluate_model(tuned_rf, X_train, y_train, X_test, y_test, 'Random Forest (Tuned)')
evaluate_model(tuned_gb, X_train, y_train, X_test, y_test, 'Gradient Boosting (Tuned)')

# Let's update our results dataframe with the tuned model scores
final_results_df = pd.DataFrame(results).T.sort_values(by='F1-Score', ascending=False)
print("\n--- Final Model Performance Comparison ---")
print(final_results_df)

**13.2. ROC Curve and AUC Score Comparison**

In [ ]:
# The ROC curve is an excellent way to visualize the performance of binary classifiers across different thresholds.
# A model with a curve closer to the top-left corner is better. The AUC score summarizes this into a single number.

plt.figure(figsize=(12, 8))

# Get predicted probabilities
y_pred_prob_xgb = tuned_xgb.predict_proba(X_test)[:, 1]
y_pred_prob_rf = tuned_rf.predict_proba(X_test)[:, 1]
y_pred_prob_gb = tuned_gb.predict_proba(X_test)[:, 1]

# Calculate ROC curve and AUC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_prob_xgb)
auc_xgb = roc_auc_score(y_test, y_pred_prob_xgb)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_prob_rf)
auc_rf = roc_auc_score(y_test, y_pred_prob_rf)

fpr_gb, tpr_gb, _ = roc_curve(y_test, y_pred_prob_gb)
auc_gb = roc_auc_score(y_test, y_pred_prob_gb)

# Plot the curves
plt.plot(fpr_xgb, tpr_xgb, label=f'Tuned XGBoost (AUC = {auc_xgb:.4f})', color='blue')
plt.plot(fpr_rf, tpr_rf, label=f'Tuned Random Forest (AUC = {auc_rf:.4f})', color='green')
plt.plot(fpr_gb, tpr_gb, label=f'Tuned Gradient Boosting (AUC = {auc_gb:.4f})', color='red')

# Plot the baseline
plt.plot([0, 1], [0, 1], 'k--', label='Baseline (AUC = 0.50)')

plt.title('ROC Curve Comparison for Tuned Models', fontsize=16)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

# 13.3. Choosing the Final Model

Based on the final results from your notebook, the **Tuned Random Forest** model emerges as the champion model. It achieves the best balance of performance, particularly with the highest F1-Score:

  * **Accuracy: 0.7442**
  * **Recall: 0.8842** (Correctly identifies over 88% of all certifiable applications)
  * **Precision: 0.7676** (When it predicts 'Certified', it is correct ~77% of the time)
  * **F1-Score: 0.8219** (The best balance between Precision and Recall)
  * **AUC Score: 0.8188** (A very strong score indicating excellent discriminative power)



Therefore, we select the **Tuned Random Forest Classifier** as our final model for providing actionable insights.

# 14\. Actionable Insights & Recommendations (as per Rubric)

**14.1. Advanced Feature Importance with SHAP**

In [ ]:
# Install SHAP library
#!pip install shap

import shap

explainer = shap.TreeExplainer(tuned_rf)
shap_values = explainer.shap_values(X_test)

print("\nGenerating SHAP Summary Plot for Tuned Random Forest...")
# The bar plot needs the full shap_values list to average importances correctly
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean Absolute SHAP Value)", fontsize=14)
plt.show()

# The beeswarm plot needs the SHAP values for the positive class (Certified)
shap.summary_plot(shap_values[1], X_test, show=False)
plt.title("SHAP Feature Impact on Model Output", fontsize=14)
plt.show()

**SHAP Plot Interpretation:** The second plot is incredibly powerful. Each dot is a single prediction.

  * **Feature Value:** Red dots mean a high value for that feature; blue dots mean a low value.
  * **SHAP Value (x-axis):** Positive values push the prediction towards 'Certified' (1), while negative values push it towards 'Denied' (0).



Example Insight: For `has_job_experience`, the red dots (value=1, meaning 'Yes') are all on the right side (positive SHAP value), strongly pushing the prediction to 'Certified'. The blue dots (value=0, meaning 'No') are on the left, pushing the prediction to 'Denied'.

# 14.2 Concluding with Key Takeaways for the Business

Based on the model's performance and the SHAP analysis, we can provide the following actionable insights and recommendations.

**Understanding the Business Cost of Errors**

Before diving into recommendations, it's crucial to understand the cost of our model's mistakes:

  * **Cost of a False Negative (Predict 'Denied' for a case that would be 'Certified'):** This is a significant opportunity cost. EasyVisa might advise a client not to proceed with a strong application, leading to client dissatisfaction, loss of trust, and potential loss of business.
  * **Cost of a False Positive (Predict 'Certified' for a case that will be 'Denied'):** This results in wasted resources. The client spends time and money on application fees for a case that was likely to fail.



Our chosen model has a very high **Recall (88.4%)** , which is excellent because it means we are very effective at minimizing the number of False Negatives. We are good at finding the "certifiable" applications, which is the primary goal.

**Key Takeaways & Insights:**

  1. **Education is Paramount:** The SHAP analysis confirms that `education_of_employee_Doctorate` and `Master's` are the top drivers. High values for these features (i.e., having these degrees) strongly push the prediction towards 'Certified'.

  2. **Experience Matters:** `has_job_experience` is the single most important binary feature. Having experience provides a strong positive push towards certification.

  3. **Continent of Origin is a Key Driver:** The model learned that applicants from **Europe** and **Asia** have a higher probability of success. The SHAP plot shows that being from Europe (`continent_Europe`=1) has a strong positive impact.

  4. **Job Profile is Crucial:** The nature of the job itself is highly influential.

     * **Salaried Positions are Preferred:** The SHAP plot shows that having a `unit_of_wage_Hour` (blue dots) has a strong negative impact, pushing predictions towards 'Denied'. Conversely, yearly wages are associated with success.
     * **Full-Time Roles are Favored:** `full_time_position` also shows a clear positive impact on getting certified.
  5. **Company Size and Age are Less Important:** As seen in the EDA and confirmed by SHAP, these features have a much lower impact on the final prediction compared to the applicant's own profile.




**Business Recommendations for EasyVisa:**

  1. **Refine Applicant Screening:** EasyVisa should advise its clients to prioritize candidates with advanced degrees (Master's or Doctorate) and demonstrable job experience. A screening checklist incorporating these top features could significantly increase the success rate.

  2. **Focus on High-Skill Roles:** Employers should be encouraged to structure their job offers as full-time, salaried positions whenever possible, as these are viewed more favorably in the certification process. This reflects a commitment to the employee and often corresponds to roles that are harder to fill with local talent.

  3. **Develop Targeted Guidance:** EasyVisa can create tailored advisory content for employers looking to hire from specific continents. For example, they could provide region-specific advice for navigating the application process for candidates from Asia and Europe.

  4. **Leverage the Predictive Model:** EasyVisa can integrate this **Tuned Random Forest model** into its internal workflow.

     * **Pre-submission Check:** Before an application is formally submitted, it can be run through the model to get a "probability of success" score.
     * **Risk Assessment:** Applications flagged by the model as having a high probability of denial can be reviewed more closely to see if the application can be strengthened before submission, saving time and resources.



By implementing these data-driven strategies, EasyVisa can enhance its service offering, improve visa approval rates for its clients, and solidify its position as a leader in the immigration services industry.